In [11]:
# ══════════════════════════════════════════════
#  FOOD_INFO :
#  density       = كثافة الأكلة (g/cm³)
#  calories_per_g = سعرات لكل جرام (kcal/g)
#
#  المصادر:
#  - Kofta        : SnapCalorie / USDA (~167 kcal / 100g)
#  - Bechamel     : LittleSpiceJar recipe nutrition (~300 kcal / 100g)
#  - Chicken      : USDA FoodData Central #171477 (~165 kcal / 100g)
#  - Mahshi       : تقدير موثوق للأكلات المصرية المحشية (~120 kcal / 100g)
#  - Taameya      : USDA falafel data (~277 kcal / 100g)
#  - Fool         : USDA fava beans cooked (~110 kcal / 100g)
#  - Molokhia     : USDA molokhia cooked (~53 kcal / 100g)
#  - Koshary      : SnapCalorie Egyptian koshary (~115 kcal / 100g)
#  - Rice         : USDA white rice cooked #168930 (~130 kcal / 100g)
# ══════════════════════════════════════════════

In [1]:
import os
import cv2
import numpy as np
import torch
from ultralytics import YOLO

In [6]:
COIN_CLASS_NAME       = "Coin"
COIN_REAL_DIAMETER_CM = 2.3

FOOD_INFO = {
    "kofta":               {"density": 1.05, "calories_per_g": 1.67},
    "macaroni_bechamel":   {"density": 0.90, "calories_per_g": 3.00},
    "Chicken":             {"density": 1.00, "calories_per_g": 1.65},
    "mahshi":              {"density": 0.88, "calories_per_g": 1.20},
    "taameya":             {"density": 0.75, "calories_per_g": 2.77},
    "fool":                {"density": 0.92, "calories_per_g": 1.10},
    "molokhia":            {"density": 0.95, "calories_per_g": 0.53},
    "koshary":             {"density": 0.85, "calories_per_g": 1.15},
    "rice":                {"density": 0.80, "calories_per_g": 1.30},
}

def load_midas():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True)
    model.to(device).eval()
    transform = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True).small_transform
    print(f"✅ MiDaS loaded on {device}")
    return model, transform, device


def get_depth_map(image_bgr, midas, transform, device):
    h, w    = image_bgr.shape[:2]
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    input_tensor = transform(img_rgb).to(device)
    with torch.no_grad():
        depth = midas(input_tensor)

    depth = torch.nn.functional.interpolate(depth.unsqueeze(1), size=(h, w), mode="bicubic", align_corners=False).squeeze().cpu().numpy()

    depth = (depth - depth.min()) / (depth.max() - depth.min())    # normalize بين 0 و 1
    return depth.astype(np.float32)

def calc_depth_scale(coin_mask, depth_map, image_shape):
    h, w = depth_map.shape[:2]
    mask_resized = cv2.resize(coin_mask.astype(np.float32), (w, h),interpolation=cv2.INTER_NEAREST).astype(bool)

    # متوسط الـ depth في منطقة الـ coin
    coin_depth_relative = float(depth_map[mask_resized].mean())

    # الـ coin سمكها ~0.2cm فعلياً — بس كـ reference للسطح نعتبرها ~2cm
    # (عمق الطبق من الكاميرا للأكل)
    COIN_REFERENCE_DEPTH_CM = 2.0
    if coin_depth_relative > 0:
        scale = COIN_REFERENCE_DEPTH_CM / coin_depth_relative
    else:
        scale = 5.0  # fallback
    return scale

def get_pixel_size_from_coin(coin_mask, target_shape=None):
    mask_to_use = coin_mask
    if target_shape is not None:
        h, w = target_shape
        mask_to_use = cv2.resize(coin_mask.astype(np.float32),(w, h),interpolation=cv2.INTER_NEAREST)

    mask_uint8  = (mask_to_use * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    largest         = max(contours, key=cv2.contourArea)
    (_, _), radius  = cv2.minEnclosingCircle(largest)
    diameter_pixels = radius * 2

    if diameter_pixels < 5:
        return None

    pixel_size_cm = COIN_REAL_DIAMETER_CM / diameter_pixels
    return pixel_size_cm


def calc_calories(food_name, food_mask, depth_map, pixel_size_cm, depth_scale):
    """
    area_cm2     = عدد الـ pixels في الـ mask  ×  pixel_size²
    avg_depth_cm = متوسط الـ depth جوه الـ mask  ×  10  (تحويل relative → cm)
    volume_cm3   = area  ×  depth
    weight_g     = volume ×  density
    calories     = weight ×  calories_per_gram
    """
    if food_name not in FOOD_INFO:
        print(f"    '{food_name}' the Food is Not in our FOOD_INFO!")
        return None

    food = FOOD_INFO[food_name]

    # ← الإضافة: resize الـ mask عشان يتطابق مع الـ depth_map
    h, w = depth_map.shape[:2]
    mask_resized = cv2.resize(food_mask.astype(np.float32),(w, h),interpolation=cv2.INTER_NEAREST)
    mask_bool = mask_resized.astype(bool)

    area_cm2     = mask_bool.sum() * (pixel_size_cm ** 2)
    avg_depth_cm = float(depth_map[mask_bool].mean()) * 5.25
    volume_cm3   = area_cm2 * avg_depth_cm
    weight_g     = volume_cm3 * food["density"]
    calories     = weight_g   * food["calories_per_g"]

    return {
        "area_cm2":   round(area_cm2,   2),
        "depth_cm":   round(avg_depth_cm, 2),
        "volume_cm3": round(volume_cm3, 2),
        "weight_g":   round(weight_g,   1),
        "calories":   round(calories,   1),
    }

# ── 5. الـ pipeline الكامل ──────────────────────
def predict(image_path):
    yolo = YOLO(r"D:\Graduation Project\Scan Food\my_yolo_results\segment\train3\weights\best.pt")
    midas, transform, device = load_midas()

    image = cv2.imread(image_path)
    if image is None:
        print("There is No Image")
        return
    results = yolo.predict(image, conf=0.25, verbose=False)
    result  = results[0]
    class_names = yolo.names

    if result.masks is None:
        print("There is Nothing Detected in the Image")
        return

    depth_map = get_depth_map(image, midas, transform, device)
    # فصّل الـ coin عن الأكل
    coin_mask  = None
    food_items = []

    for i in range(len(result.masks)):
        mask       = result.masks.data[i].cpu().numpy()
        class_name = class_names[int(result.boxes.cls[i])]
        confidence = float(result.boxes.conf[i])

        if class_name == COIN_CLASS_NAME:
            coin_mask = mask
            print(f" Coin Detected (confidence: {confidence:.0%})")
        else:
            food_items.append((class_name, mask, confidence))

    if coin_mask is None:
        print("There is No Coin in the Picture")
        return

    pixel_size_cm = get_pixel_size_from_coin(coin_mask, target_shape=image.shape[:2])
    depth_scale = calc_depth_scale(coin_mask, depth_map, image.shape[:2])
    if pixel_size_cm is None:
        print("There is an Error in Calculating Volume of the coin")
        return
    print(f" each pixel = {pixel_size_cm:.4f} cm")


    print("\n─── Results  ───────────────────────────────")
    total_calories = 0
    for food_name, mask, confidence in food_items:
        data = calc_calories(food_name, mask, depth_map, pixel_size_cm, depth_scale)
        if data:
            total_calories += data["calories"]
            print(f"\n  🍽️  {food_name}  ({confidence:.0%})")
            print(f"     Area : {data['area_cm2']} cm²")
            print(f"     Density   : {data['depth_cm']} cm")
            print(f"     Volume   : {data['volume_cm3']} cm³")
            print(f"     Weight   : {data['weight_g']} g")
            print(f"     Calories : {data['calories']} kcal")
            print("────────────────────────────────────────────")

    print(f"\n   Total Calories : {total_calories:.0f} kcal")
    # Show for show and plot for save
    result.show()
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    cv2.imwrite(os.path.join(".", "D:\Computer Vision\Segmentation" ,"output.jpg"), result.plot())

if __name__ == "__main__":
    predict("D:\Computer Vision\Segmentation\WhatsApp Image 2026-05-04 at 3.25.17 PM.jpeg")

Using cache found in C:\Users\muhaned/.cache\torch\hub\intel-isl_MiDaS_master


Loading weights:  None


Using cache found in C:\Users\muhaned/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\muhaned/.cache\torch\hub\intel-isl_MiDaS_master


✅ MiDaS loaded on cpu
 Coin Detected (confidence: 86%)
 each pixel = 0.0172 cm

─── Results  ───────────────────────────────

  🍽️  rice  (96%)
     Area : 132.22 cm²
     Density   : 3.63 cm
     Volume   : 480.04 cm³
     Weight   : 384.0 g
     Calories : 499.2 kcal
────────────────────────────────────────────

  🍽️  Chicken  (35%)
     Area : 37.23 cm²
     Density   : 3.1 cm
     Volume   : 115.59 cm³
     Weight   : 115.6 g
     Calories : 190.7 kcal
────────────────────────────────────────────

   Total Calories : 690 kcal
